# 5.4 阶段二：BN1 简单分核

单核跑对之后，第一个性能问题显而易见：**一颗 AI Core 干活，其他核全在围观**。阶段二引入多核——按 B、N 两轴切分任务，一个 (batch, head) 组合交给一个核。

## 1. 核间切块的第一原则

> **核间切分最好不要有数据依赖。**

为什么？SPMD 多核并行（第 3 章 3.6 节）的并行条件就是「多核间的数据可以独立运算」。有依赖就必须核间同步，一次同步的代价可能吃掉并行收益。

FA 各轴的依赖情况：

| 轴 | 切开后有依赖吗 | 原因 |
|--|--|--|
| B（batch） | ❌ 无 | 不同样本完全独立 |
| N（head） | ❌ 无 | 不同头的 QKV 互不相干 |
| S1（Q 序列） | ❌ 无 | 不同 Q 块的输出行互不相干 |
| S2（KV 序列） | ✅ 有 | Softmax 沿 S2 归一化，分母要见全部 KV（第 3 章 3.6 节） |

所以安全的核间切分轴是 **B、N、S1**；S2 永远留给核内循环——这条原则将贯穿第 5~9 章。

## 2. BN1 分核

BN1 分核 = 按 **B × N** 的任务总数分核（每个 (b, n) 一个核，不足时少用核）：

```text
totalSize = B × N
例：B=2, N=20 → 40 个任务
   有 40 个核 → 每核 1 个 (b, n) → blockDim = 40
   只有 20 个核 → 每核 2 个 (b, n) → blockDim = 20
```

Kernel 侧的映射：

```cpp
if (block_idx >= blockDim) return;

// BN1：核 i 负责第 i 个 (b, n) 组合（或连续领多组）
int64_t coreTaskIdx = block_idx;
int64_t bIdx = coreTaskIdx / nSize;      // 先按 N 展开
int64_t nIdx = coreTaskIdx % nSize;

// GM 偏移：BSND 排布下按 (b, n) 计算各自的 QKV 基地址
int64_t qOffset = (bIdx * s1Size * nSize + nIdx) * dSize;
// 每个核把「自己那份」QKV 搬进片上，四拍循环（复用阶段一代码）
ProcessSingleTask(bIdx, nIdx, ...);
```

> Tiling 侧对应改动：`totalSize = B × N`，`CalcTschBlockDim` 算出 blockDim——正是第 3 章 3.5 节 `SetMultiCoreParams` 的极简版。

## 3. 效果与局限

| 维度 | 收益 |
|--|--|
| 并行度 | B×N 倍提速（B×N ≤ 核数时） |
| 改动量 | 只改 Tiling 的切分 + Kernel 的 offset 映射 |

**局限**：任务粒度太粗。

- 大模型 B=1、N=32 的 Decode 场景：totalSize = 32，而核数有几十个——一半核闲置；
- B、N 都小时任务数不足，**切不动了**。

## 4. 出路预告：FlashDecode

推理 Generation（Decode）阶段逐 token 生成，利用 KV Cache 时每次实际的 queries token 数为 **1**，已经无法通过 Q、BS 进一步并行。

> **既然 Q 和 BS 无法进一步并行了，那么对 K、V 进行并行就是 FlashDecode 的思路。**

把 S2 也切给多核（每核算一段部分 softmax 结果），最后做一次**跨核归并**（每核的 m/l 部分状态合并、部分和重标定相加）。它打破了「S2 不能核间切」的一般原则——代价是引入受控的核间通信与二次归并 Kernel。这是第 9 章负载均衡的伏笔。

## 小测验

1. 为什么核间切块要「无数据依赖」？有依赖时付出什么代价？
2. BN1 下 B=1、N=8、40 个核可用，blockDim 是多少？多少核在空转？
3. FlashDecode 打破了哪条切分原则？用什么代价换来的？